# 02 – MLP modelio eksperimentai

## Darbo tikslas

Šiame užrašinėje tiriamas **daugiasluoksnis perceptronas (MLP, Multi-Layer Perceptron)**, skirtas kitos valandos dviračių paklausai prognozuoti.

$$
X_t \longrightarrow \hat{y}_{t+1}
$$

čia:

- $X_t$ – laiko momentu $t$ modeliui prieinami požymiai;
- $\hat{y}_{t+1}$ – prognozuojama kitos valandos dviračių paklausa.

Šio užrašinėje tikslas – paaiškinti MLP veikimą, atlikti pradinę hiperparametrų paiešką, po jos tikslinti perspektyviausią parametrų sritį ir galutinę MLP konfigūraciją pasirinkti **tik pagal DEV rezultatus**.

> Galutinis TEST šiame užrašinėje nenaudojamas nei hiperparametrams parinkti, nei modeliui tobulinti.

In [ ]:
from pathlib import Path
import sys, warnings, json
warnings.filterwarnings("ignore")
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / "hour.csv"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
from src.common import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import ParameterGrid
work=load_prepare(DATA_PATH)
FEATURES=feature_columns(work)
dev,test=split_dev_test(work)
folds=rolling_origin_folds(dev)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from IPython.display import Markdown, display

## Kodėl pasirinktas MLP metodas?

MLP pasirinktas kaip vienas iš intelektualiųjų regresijos metodų, nes prognozuojama dviračių paklausa gali priklausyti nuo kelių tarpusavyje susijusių požymių ir jų netiesinių sąryšių. MLP gali mokytis tokių sąryšių iš duomenų, todėl šiame darbe jis naudojamas kaip alternatyva sezoniniam atskaitos metodui ir kitiems intelektualiesiems modeliams.

MLP rezultatai nėra vertinami atskirai nuo kitų metodų. Galutiniame darbo etape visi pasirinkti modeliai bus lyginami naudojant tą patį TEST laikotarpį ir tas pačias vertinimo metrikas.

Šiame užrašinėje tikrinama ne tai, ar MLP teoriškai turėtų būti geriausias metodas, o tai, **kokį rezultatą jis realiai pasiekia vienodomis eksperimento sąlygomis**.

## 1. MLP veikimo principas ir ryšys su kodu

MLP neuroniniame tinkle kiekvienas neuronas pirmiausia apskaičiuoja svertinę įėjimų sumą:

$$
z_j
=
\sum_{i=1}^{n}
w_{ji}x_i
+
b_j
$$

čia:

- $z_j$ – $j$-ojo neurono svertinė įėjimų suma;
- $x_i$ – $i$-asis įvesties požymis;
- $w_{ji}$ – svoris, jungiantis $i$-ąjį įėjimą su $j$-uoju neuronu;
- $b_j$ – $j$-ojo neurono poslinkis (*poslinkis (bias)*);
- $n$ – neurono įėjimų skaičius.

Apskaičiavus svertinę sumą, pritaikoma aktyvacijos funkcija:

$$
a_j
=
f(z_j)
$$

čia:

- $a_j$ – $j$-ojo neurono išėjimas;
- $f(\cdot)$ – aktyvacijos funkcija;
- $z_j$ – neurono svertinė įėjimų suma.

Šiame darbe paslėptuose MLP sluoksniuose naudojama `ReLU` aktyvacijos funkcija:

$$
f(z)
=
\max(0,z)
$$

Tai reiškia, kad kai $z>0$, neurono išėjimas lygus $z$, o kai $z\leq0$, neurono išėjimas lygus $0$.

### Formulės ryšys su programiniu kodu

Tinklo struktūrą kode nusako:

```python
MLPRegressor(hidden_layer_sizes=hidden, ...)
```

Pavyzdžiui:

```python
hidden = (64, 64, 32)
```

reiškia, kad MLP turi tris paslėptus sluoksnius, kuriuose yra atitinkamai **64, 64 ir 32 neuronai**.

Eksperimente derinami šie hiperparametrai:

- `hidden_layer_sizes` – paslėptų sluoksnių skaičius ir neuronų skaičius juose;
- `alpha` – L2 reguliarizavimo stiprumas;
- `learning_rate_init` – pradinė mokymosi sparta.

L2 reguliarizavimo principą galima užrašyti:

$$
L_{\mathrm{reg}}
=
L
+
\alpha
\sum_j w_j^2
$$

čia:

- $L_{\mathrm{reg}}$ – nuostolio funkcija su reguliarizavimu;
- $L$ – pagrindinė modelio nuostolio funkcija;
- $\alpha$ – L2 reguliarizavimo koeficientas;
- $w_j$ – modelio svoriai.

Prieš MLP mokymą naudojama `Pipeline` grandinė:

1. `SimpleImputer(strategy="median")` – užpildo trūkstamas požymių reikšmes;
2. `StandardScaler()` – standartizuoja požymius;
3. `MLPRegressor(...)` – apmoko neuroninį tinklą.

Kiekviename DEV skaidinyje `SimpleImputer` ir `StandardScaler` pritaikomi tik mokymo duomenims, todėl validavimo duomenų informacija nepanaudojama iš anksto.

### Kodėl prieš MLP atliekamas požymių standartizavimas?

MLP mokymo metu vienu metu naudojami skirtingo mastelio požymiai. Kad didesnės skaitinės reikšmės vien dėl savo mastelio neturėtų neproporcingos įtakos mokymui, požymiai standartizuojami naudojant `StandardScaler`.

Standartizavimo principas:

$$
x_i^{\mathrm{std}}
=
\frac{x_i-\mu_i}{\sigma_i}
$$

čia:

- $x_i$ – pradinė požymio reikšmė;
- $\mu_i$ – požymio vidurkis, apskaičiuotas iš mokymo duomenų;
- $\sigma_i$ – požymio standartinis nuokrypis, apskaičiuotas iš mokymo duomenų;
- $x_i^{\mathrm{std}}$ – standartizuota požymio reikšmė.

Svarbu, kad kiekviename `slenkančios pradžios (rolling-origin)` skaidinyje `StandardScaler` parametrai $\mu_i$ ir $\sigma_i$ apskaičiuojami **tik iš mokymo dalies**. Taip validavimo laikotarpio informacija nepatenka į modelio mokymo procesą.

## 2. Duomenys ir vienodos eksperimento sąlygos

Naudojama ta pati duomenų grandinė kaip 01 užrašinėje:

- tas pats požymių rinkinys;
- ta pati DEV dalis;
- tie patys 3 chronologiniai `slenkančios pradžios (rolling-origin)` skaidiniai;
- tos pačios MAE, RMSE, Peak MAE ir PUR metrikos;
- piko riba apskaičiuojama tik iš konkretaus mokymo skaidinio;
- naudojama fiksuota `SEED` reikšmė.

Taip užtikrinama, kad skirtingos MLP konfigūracijos būtų palygintos vienodomis sąlygomis.

In [ ]:
work = load_prepare(DATA_PATH)
FEATURES = feature_columns(work)
dev, test = split_dev_test(work)
folds = rolling_origin_folds(dev)

report = leakage_report(work, dev, test, FEATURES)
assert report["chronological_sorted"]
assert report["target_after_feature_time"]
assert not report["casual_in_features"]
assert not report["registered_in_features"]
assert report["dev_before_test"]

print("Požymių skaičius:", len(FEATURES))
print("DEV stebėjimų skaičius:", len(dev))
print("Rezervuoto TEST stebėjimų skaičius:", len(test))
print("DEV rolling-origin skaidinių skaičius:", len(folds))
print("Pagrindinės duomenų nutekėjimo patikros PRAEITOS.")

## 3. Vienoda MLP vertinimo funkcija

Kiekvienai konfigūracijai atliekama ta pati procedūra: modelis mokomas visuose DEV `slenkančios pradžios (rolling-origin)` skaidiniuose, kiekviename skaidinyje iš naujo pritaikomi `SimpleImputer` ir `StandardScaler`, o rezultatai vertinami tomis pačiomis metrikomis.

In [ ]:
def evaluate_mlp_config(hidden, alpha, lr, return_last=False):
    rows = []
    last_data = None

    for fold_id, (tri, vai) in enumerate(folds, 1):
        tr = dev.iloc[tri].copy()
        va = dev.iloc[vai].copy()

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPRegressor(
                hidden_layer_sizes=hidden,
                activation="relu",
                solver="adam",
                alpha=alpha,
                learning_rate_init=lr,
                max_iter=400,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=20,
                random_state=SEED
            ))
        ])

        model.fit(tr[FEATURES], tr["target"])
        pred = model.predict(va[FEATURES])
        peak = float(tr["target"].quantile(PEAK_Q))

        rows.append({"fold": fold_id, **metrics(va["target"], pred, peak)})

        if return_last and fold_id == len(folds):
            last_data = {"train": tr, "valid": va, "pred": pred,
                         "model": model, "peak_threshold": peak}

    fold_df = pd.DataFrame(rows)
    summary = {
        "hidden": hidden, "alpha": alpha, "lr": lr,
        "MAE": fold_df["MAE"].mean(),
        "RMSE": fold_df["RMSE"].mean(),
        "Peak_MAE": fold_df["Peak_MAE"].mean(),
        "PUR": fold_df["PUR"].mean(),
        "N": fold_df["N"].mean(),
        "N_peak": fold_df["N_peak"].mean()
    }
    return (summary, fold_df, last_data) if return_last else summary

## 4. Geriausių MLP parametrų paieška – pirmasis etapas

Tikrinamos 3 tinklo struktūros, 2 `alpha` reikšmės ir 2 mokymosi spartos:

$$
3\times2\times2=12
$$

konfigūracijų. Kadangi kiekviena vertinama 3 DEV skaidiniuose:

$$
12\times3=36
$$

MLP mokymai.

Pagrindinis parinkimo kriterijus – **mažiausia vidutinė DEV MAE**. Kartu stebimos RMSE, Peak MAE ir PUR, kad būtų matoma modelio elgsena ir piko valandomis.

### Kodėl pagrindinis parametrų parinkimo kriterijus yra MAE?

Pagrindiniu hiperparametrų parinkimo kriterijumi pasirinkta **vidutinė absoliuti klaida (MAE)**:

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum_{i=1}^{N}
\left|
y_i-\hat{y}_i
\right|
$$

čia:

- $y_i$ – faktinė dviračių paklausa;
- $\hat{y}_i$ – MLP prognozuota paklausa;
- $N$ – vertinamų stebėjimų skaičius.

MAE tiesiogiai parodo, kiek nuomų vidutiniškai skiriasi prognozė nuo faktinės paklausos, todėl šią metriką lengva interpretuoti praktiniame uždavinyje. **Mažesnė MAE reikšmė yra geresnė.**

Tačiau vien MAE nepakanka visam modelio elgesiui apibūdinti. Todėl kartu pateikiamos:

- **RMSE** – stipriau reaguoja į dideles prognozavimo klaidas;
- **Peak MAE** – parodo absoliučią klaidą piko valandomis;
- **PUR** – parodo, kokia piko valandų dalis buvo prognozuota per maža.

Taigi MLP konfigūracija parenkama pagal DEV MAE, o kitos metrikos naudojamos papildomai įvertinti pasirinktos konfigūracijos elgseną.

In [ ]:
initial_configs = [
    {"hidden": (64,32), "alpha": 0.0001, "lr": 0.0005},
    {"hidden": (64,32), "alpha": 0.0001, "lr": 0.0010},
    {"hidden": (128,64), "alpha": 0.0001, "lr": 0.0005},
    {"hidden": (128,64), "alpha": 0.0001, "lr": 0.0010},
    {"hidden": (64,64,32), "alpha": 0.0001, "lr": 0.0005},
    {"hidden": (64,64,32), "alpha": 0.0001, "lr": 0.0010},
    {"hidden": (64,32), "alpha": 0.0010, "lr": 0.0005},
    {"hidden": (64,32), "alpha": 0.0010, "lr": 0.0010},
    {"hidden": (128,64), "alpha": 0.0010, "lr": 0.0005},
    {"hidden": (128,64), "alpha": 0.0010, "lr": 0.0010},
    {"hidden": (64,64,32), "alpha": 0.0010, "lr": 0.0005},
    {"hidden": (64,64,32), "alpha": 0.0010, "lr": 0.0010},
]

initial_results = pd.DataFrame([
    evaluate_mlp_config(**cfg) for cfg in initial_configs
]).sort_values("MAE").reset_index(drop=True)

display(initial_results)
best_initial = initial_results.iloc[0]

display(Markdown(f'''
### Geriausia pirmojo etapo konfigūracija

- struktūra: **{best_initial["hidden"]}**;
- `alpha`: **{best_initial["alpha"]:.6g}**;
- `learning_rate_init`: **{best_initial["lr"]:.6g}**;
- DEV MAE: **{best_initial["MAE"]:.2f}**;
- DEV RMSE: **{best_initial["RMSE"]:.2f}**;
- DEV Peak MAE: **{best_initial["Peak_MAE"]:.2f}**;
- DEV PUR: **{best_initial["PUR"]:.4f}**.

Konfigūracija parinkta tik pagal DEV. TEST nenaudotas.
'''))

In [ ]:
plt.figure(figsize=(11,4))
labels = [f"{tuple(r.hidden)}\nα={r.alpha:g}, lr={r.lr:g}" for r in initial_results.itertuples()]
plt.bar(range(len(initial_results)), initial_results["MAE"])
plt.xticks(range(len(initial_results)), labels, rotation=45, ha="right")
plt.title("Pradinės MLP hiperparametrų paieškos DEV MAE")
plt.xlabel("MLP konfigūracija")
plt.ylabel("MAE")
plt.tight_layout()
plt.show()

## 5. Geriausių MLP parametrų paieška – antrasis tikslinimo etapas

Pirmasis etapas parodo perspektyviausią parametrų sritį. Toliau atliekama **siauresnė paieška aplink realiai geriausią pirmojo etapo konfigūraciją**.

Tai nėra abliacija. Tai antras hiperparametrų paieškos etapas.

Automatiškai tikrinama:

- pradinė geriausia konfigūracija;
- perpus mažesnė ir dvigubai didesnė `alpha`;
- 0,7 karto mažesnė ir 1,5 karto didesnė mokymosi sparta;
- šiek tiek platesnė ir siauresnė tinklo struktūra.

Visos konfigūracijos vėl vertinamos tais pačiais DEV skaidiniais. TEST nenaudojamas.

In [ ]:
best_hidden = tuple(best_initial["hidden"])
best_alpha = float(best_initial["alpha"])
best_lr = float(best_initial["lr"])

wider_hidden = tuple(max(8, int(round(v*1.5))) for v in best_hidden)
narrower_hidden = tuple(max(8, int(round(v*0.75))) for v in best_hidden)

refined_configs = [
    {"hidden": best_hidden, "alpha": best_alpha, "lr": best_lr},
    {"hidden": best_hidden, "alpha": best_alpha*0.5, "lr": best_lr},
    {"hidden": best_hidden, "alpha": best_alpha*2.0, "lr": best_lr},
    {"hidden": best_hidden, "alpha": best_alpha, "lr": best_lr*0.7},
    {"hidden": best_hidden, "alpha": best_alpha, "lr": best_lr*1.5},
    {"hidden": wider_hidden, "alpha": best_alpha, "lr": best_lr},
    {"hidden": narrower_hidden, "alpha": best_alpha, "lr": best_lr},
]

# pašalinamos galimos pasikartojančios konfigūracijos
seen, unique = set(), []
for cfg in refined_configs:
    key=(cfg["hidden"],cfg["alpha"],cfg["lr"])
    if key not in seen:
        seen.add(key); unique.append(cfg)
refined_configs=unique

display(pd.DataFrame(refined_configs))

In [ ]:
refined_results = pd.DataFrame([
    evaluate_mlp_config(**cfg) for cfg in refined_configs
]).sort_values("MAE").reset_index(drop=True)

display(refined_results)
best_refined = refined_results.iloc[0]
improvement = float(best_initial["MAE"]) - float(best_refined["MAE"])

display(Markdown(f'''
### Hiperparametrų tikslinimo rezultatas

- struktūra: **{best_refined["hidden"]}**;
- `alpha`: **{best_refined["alpha"]:.6g}**;
- `learning_rate_init`: **{best_refined["lr"]:.6g}**;
- DEV MAE: **{best_refined["MAE"]:.2f}**;
- DEV RMSE: **{best_refined["RMSE"]:.2f}**;
- DEV Peak MAE: **{best_refined["Peak_MAE"]:.2f}**;
- DEV PUR: **{best_refined["PUR"]:.4f}**.

MAE pagerėjimas, palyginti su pirmojo etapo geriausiu rezultatu: **{improvement:+.2f}** nuomos.

Jei reikšmė teigiama, MAE sumažėjo. Jei lygi nuliui arba neigiama, papildomas tikslinimas MAE nepagerino.
'''))

### Ką parodo antrasis parametrų paieškos etapas?

Pirmasis etapas skirtas platesnei parametrų sričiai patikrinti, o antrasis – tiksliau ištirti sritį aplink geriausią pirmojo etapo rezultatą.

Jeigu antrame etape gaunama mažesnė DEV MAE, galima teigti, kad siauresnis parametrų tikslinimas pagerino MLP rezultatą. Jeigu MAE nesumažėja, tai reiškia, kad tikrintoje parametrų srityje papildomas tikslinimas geresnės konfigūracijos nerado.

Abiem atvejais paliekamas realiai gautas rezultatas – parametrai nekeičiami pagal TEST duomenis.

In [ ]:
comparison = pd.DataFrame({
    "Etapas":["Pradinė paieška","Tikslinimas"],
    "MAE":[best_initial["MAE"],best_refined["MAE"]],
    "RMSE":[best_initial["RMSE"],best_refined["RMSE"]],
    "Peak_MAE":[best_initial["Peak_MAE"],best_refined["Peak_MAE"]],
    "PUR":[best_initial["PUR"],best_refined["PUR"]]
})
display(comparison)

plt.figure(figsize=(8,4))
plt.bar(comparison["Etapas"],comparison["MAE"])
plt.title("Geriausios MLP konfigūracijos MAE prieš ir po tikslinimo")
plt.ylabel("MAE")
plt.tight_layout()
plt.show()

## 6. Geriausių MLP parametrų parinkimas pagal DEV

Galutinė MLP konfigūracija parenkama iš visų abiejų etapų bandymų pagal mažiausią vidutinę DEV MAE. RMSE, Peak MAE ir PUR paliekamos kaip papildomos kokybės charakteristikos.

TEST šiame etape vis dar neliečiamas.

In [ ]:
all_results = pd.concat([
    initial_results.assign(stage="Pradinė paieška"),
    refined_results.assign(stage="Tikslinimas")
],ignore_index=True)

all_results = all_results.sort_values("MAE").drop_duplicates(
    subset=["hidden","alpha","lr"]
).reset_index(drop=True)

best_mlp=all_results.iloc[0]

display(Markdown(f'''
### Galutinė MLP konfigūracija pagal DEV

- etapas: **{best_mlp["stage"]}**;
- struktūra: **{best_mlp["hidden"]}**;
- `alpha`: **{best_mlp["alpha"]:.6g}**;
- `learning_rate_init`: **{best_mlp["lr"]:.6g}**;
- DEV MAE: **{best_mlp["MAE"]:.2f}**;
- DEV RMSE: **{best_mlp["RMSE"]:.2f}**;
- DEV Peak MAE: **{best_mlp["Peak_MAE"]:.2f}**;
- DEV PUR: **{best_mlp["PUR"]:.4f}**.

Ši konfigūracija užfiksuojama vėlesniam bendram TEST palyginimui.
'''))

## 7. MLP palyginimas su sezoniniu atskaitos metodu DEV rinkinyje

MLP rezultatą svarbu vertinti ne tik atskirai, bet ir palyginti su paprastesniu sezoniniu atskaitos metodu. Atskaitos metodas naudoja tos pačios savaitės dienos ir valandos ankstesnių savaičių paklausą.

Kad palyginimas būtų korektiškas, abu metodai vertinami tais pačiais DEV `slenkančios pradžios (rolling-origin)` skaidiniais ir tomis pačiomis metrikomis.

Santykinį MAE pokytį galima apskaičiuoti:

$$
\Delta_{\mathrm{MAE}}
=
\frac{
\mathrm{MAE}_{\mathrm{baseline}}
-
\mathrm{MAE}_{\mathrm{MLP}}
}{
\mathrm{MAE}_{\mathrm{baseline}}
}
\cdot 100\%
$$

čia:

- $\mathrm{MAE}_{\mathrm{baseline}}$ – sezoninio atskaitos metodo vidutinė DEV MAE;
- $\mathrm{MAE}_{\mathrm{MLP}}$ – pasirinktos MLP konfigūracijos vidutinė DEV MAE;
- $\Delta_{\mathrm{MAE}}$ – santykinis MAE sumažėjimas procentais.

Teigiama $\Delta_{\mathrm{MAE}}$ reikšmė reiškia, kad MLP MAE yra mažesnė už atskaitos metodo MAE. Neigiama reikšmė reikštų, kad MLP pagal MAE atskaitos metodo nepagerino.

In [ ]:
baseline_rows = []

for fold_id, (tri, vai) in enumerate(folds, 1):
    tr = dev.iloc[tri].copy()
    va = dev.iloc[vai].copy()

    baseline_pred = seasonal_baseline(
        tr,
        va,
        k_weeks=4
    )

    peak_threshold = float(tr["target"].quantile(PEAK_Q))
    row = {
        "fold": fold_id,
        **metrics(
            va["target"],
            baseline_pred,
            peak_threshold
        )
    }
    baseline_rows.append(row)

baseline_folds = pd.DataFrame(baseline_rows)

baseline_mean = baseline_folds[["MAE", "RMSE", "Peak_MAE", "PUR"]].mean()
mlp_mean = pd.Series({
    "MAE": float(best_mlp["MAE"]),
    "RMSE": float(best_mlp["RMSE"]),
    "Peak_MAE": float(best_mlp["Peak_MAE"]),
    "PUR": float(best_mlp["PUR"])
})

baseline_vs_mlp = pd.DataFrame({
    "Sezoninis atskaitos metodas": baseline_mean,
    "MLP": mlp_mean
}).T

display(baseline_vs_mlp)

mae_change_pct = (
    (baseline_mean["MAE"] - mlp_mean["MAE"])
    / baseline_mean["MAE"]
    * 100
)

if mae_change_pct > 0:
    comparison_text = (
        f"MLP DEV MAE yra {mae_change_pct:.1f} % mažesnė už "
        "sezoninio atskaitos metodo MAE."
    )
elif mae_change_pct < 0:
    comparison_text = (
        f"MLP DEV MAE yra {abs(mae_change_pct):.1f} % didesnė už "
        "sezoninio atskaitos metodo MAE."
    )
else:
    comparison_text = "MLP ir sezoninio atskaitos metodo DEV MAE sutampa."

display(Markdown(f'''
### Palyginimo rezultatas

- sezoninio atskaitos metodo DEV MAE: **{baseline_mean["MAE"]:.2f}**;
- MLP DEV MAE: **{mlp_mean["MAE"]:.2f}**;
- **{comparison_text}**

Šis palyginimas atliekamas DEV rinkinyje. Galutinis visų modelių palyginimas su TEST duomenimis bus atliekamas tik bendrame galutinių rezultatų Notebook.
'''))

plot_df = baseline_vs_mlp[["MAE", "RMSE"]]

x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(x - width/2, plot_df["MAE"], width, label="MAE")
plt.bar(x + width/2, plot_df["RMSE"], width, label="RMSE")
plt.xticks(x, plot_df.index)
plt.ylabel("Klaida, nuomų skaičius")
plt.title("MLP ir sezoninio atskaitos metodo palyginimas DEV rinkinyje")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Pasirinktos MLP konfigūracijos stabilumas DEV skaidiniuose

Pasirinkta konfigūracija dar kartą paleidžiama tuose pačiuose DEV skaidiniuose. Tai nėra naujas parametrų derinimas – tik parodoma, kaip galutinė konfigūracija elgiasi skirtingais laikotarpiais.

In [ ]:
best_summary,best_folds,best_last=evaluate_mlp_config(
    hidden=tuple(best_mlp["hidden"]),
    alpha=float(best_mlp["alpha"]),
    lr=float(best_mlp["lr"]),
    return_last=True
)
display(best_folds)

x=np.arange(len(best_folds)); width=0.35
plt.figure(figsize=(9,4))
plt.bar(x-width/2,best_folds["MAE"],width,label="MAE")
plt.bar(x+width/2,best_folds["RMSE"],width,label="RMSE")
plt.xticks(x,[f"DEV {int(v)}" for v in best_folds["fold"]])
plt.xlabel("Chronologinis DEV skaidinys")
plt.ylabel("Klaida, nuomų skaičius")
plt.title("Pasirinktos MLP konfigūracijos stabilumas DEV skaidiniuose")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Faktinės ir prognozuotos paklausos palyginimas

Paskutiniame DEV skaidinyje palyginama faktinė paklausa ir pasirinktos MLP konfigūracijos prognozė. Tai leidžia vertinti ne tik bendras metrikas, bet ir modelio elgseną laike.

In [ ]:
va=best_last["valid"]; pred=best_last["pred"]
n=min(168,len(va))
plt.figure(figsize=(13,4))
plt.plot(va["target_time"].iloc[:n],va["target"].iloc[:n],label="Faktinė paklausa",linewidth=2)
plt.plot(va["target_time"].iloc[:n],pred[:n],label="MLP prognozė")
plt.title("MLP: faktinė ir prognozuota paklausa paskutiniame DEV skaidinyje")
plt.xlabel("Laikas"); plt.ylabel("Nuomų skaičius"); plt.legend()
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

last_metrics=metrics(va["target"],pred,best_last["peak_threshold"])
display(pd.Series(last_metrics,name="Reikšmė"))

display(Markdown(f'''
### Paskutinio DEV skaidinio rezultatų įvertinimas

- **MAE = {last_metrics["MAE"]:.2f}**;
- **RMSE = {last_metrics["RMSE"]:.2f}**;
- **Peak MAE = {last_metrics["Peak_MAE"]:.2f}**;
- **PUR = {last_metrics["PUR"]:.4f}** – **{last_metrics["PUR"]*100:.1f} %** piko valandų paklausa prognozuota per maža.

Įvertinta **{int(last_metrics["N"])} valandų**, iš jų **{int(last_metrics["N_peak"])}** – piko valandos.
'''))

## 10. MLP mokymosi eigos analizė

Vien galutinės prognozavimo metrikos neparodo, kaip modelis buvo apmokytas. Todėl papildomai analizuojama pasirinktos MLP konfigūracijos mokymosi eiga.

`MLPRegressor` mokymo metu išsaugo nuostolio funkcijos reikšmes `loss_curve_`. Ši kreivė leidžia stebėti, kaip keičiantis iteracijų skaičiui keitėsi modelio mokymo nuostolis.

Mokymo nuostolio mažėjimas rodo, kad optimizavimo algoritmas mokymo metu koregavo tinklo svorius taip, kad modelio prognozės geriau atitiktų mokymo duomenis. Tačiau vien mažėjanti mokymo nuostolio reikšmė savaime neįrodo gero modelio apibendrinimo, todėl galutinė modelio kokybė vertinama atskirai DEV duomenimis.

Šiame darbe naudojamas ir **ankstyvasis stabdymas** (`early_stopping=True`). Jam skirta dalis mokymo duomenų:

```python
validation_fraction = 0.1
```

o parametras

```python
n_iter_no_change = 20
```

nurodo, kiek iš eilės iteracijų leidžiama negauti pakankamo validavimo rezultato pagerėjimo prieš sustabdant mokymą.

Didžiausias leistinas iteracijų skaičius yra:

```python
max_iter = 400
```

Todėl realiai atliktų iteracijų skaičius $N_{\mathrm{iter}}$ gali būti mažesnis už 400:

$$
N_{\mathrm{iter}} \leq 400
$$

Ankstyvasis stabdymas naudojamas tam, kad modelio nereikėtų mokyti daugiau iteracijų, kai validavimo rezultatas nebegerėja. Tai taip pat padeda riboti perteklinį prisitaikymą prie mokymo duomenų.

In [ ]:
# Pasirinktos MLP konfigūracijos modelis iš paskutinio DEV skaidinio
trained_mlp = best_last["model"].named_steps["model"]

loss_curve = trained_mlp.loss_curve_
n_iter = trained_mlp.n_iter_

print(f"Realiai atliktų MLP iteracijų skaičius: {n_iter}")
print(f"Nustatytas didžiausias iteracijų skaičius: {trained_mlp.max_iter}")

plt.figure(figsize=(9, 4))
plt.plot(
    range(1, len(loss_curve) + 1),
    loss_curve
)
plt.xlabel("Iteracija")
plt.ylabel("Mokymo nuostolis")
plt.title("Pasirinktos MLP konfigūracijos mokymosi kreivė")
plt.tight_layout()
plt.show()

if n_iter < trained_mlp.max_iter:
    stop_text = (
        f"Mokymas baigtas po **{n_iter} iteracijų**, nors didžiausias "
        f"leistinas skaičius buvo **{trained_mlp.max_iter}**. "
        "Tai rodo, kad mokymas buvo sustabdytas anksčiau, nelaukiant "
        "maksimalaus iteracijų skaičiaus."
    )
else:
    stop_text = (
        f"Modelis atliko visas **{trained_mlp.max_iter} iteracijų**. "
        "Šiame mokyme ankstyvasis stabdymas nesustabdė mokymo anksčiau."
    )

display(Markdown(f'''
### Mokymosi eigos rezultatas

{stop_text}

Galutinė modelio kokybė vis tiek vertinama pagal atskirus DEV duomenis, o ne pagal mokymo nuostolį.
'''))

## 11. Faktinės ir prognozuotos paklausos sklaidos diagrama

Laiko grafikas parodo prognozės kitimą chronologine tvarka, tačiau papildoma sklaidos diagrama leidžia tiesiogiai palyginti kiekvieną faktinę ir prognozuotą reikšmę.

Idealios prognozės atveju:

$$
\hat{y}_i=y_i
$$

todėl visi taškai būtų išsidėstę ties įstrižaine $\hat{y}=y$.

Kuo arčiau šios linijos yra taškai, tuo mažesnis faktinės ir prognozuotos paklausos skirtumas. Taškai žemiau idealios linijos reiškia paklausos neįvertinimą, o taškai aukščiau jos – pervertinimą.

In [ ]:
y_true = va["target"].to_numpy()
y_pred = np.asarray(pred)

lower = min(y_true.min(), y_pred.min())
upper = max(y_true.max(), y_pred.max())

plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred, alpha=0.45)
plt.plot(
    [lower, upper],
    [lower, upper],
    linestyle="--",
    label=r"Ideali prognozė: $\hat{y}=y$"
)

plt.xlabel("Faktinė paklausa")
plt.ylabel("Prognozuota paklausa")
plt.title("MLP: faktinės ir prognozuotos paklausos palyginimas")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Prognozavimo paklaidų pasiskirstymas

Papildomai vertinama ne absoliuti klaida, o **pasirašyta prognozavimo paklaida**:

$$
e_i
=
y_i-\hat{y}_i
$$

čia:

- $e_i$ – prognozavimo paklaida;
- $y_i$ – faktinė paklausa;
- $\hat{y}_i$ – prognozuota paklausa.

Pagal pasirinktą ženklų apibrėžimą:

- kai $e_i>0$, faktinė paklausa buvo didesnė už prognozuotą – modelis paklausą **neįvertino**;
- kai $e_i<0$, prognozė buvo didesnė už faktinę paklausą – modelis paklausą **pervertino**;
- kai $e_i=0$, prognozė tiksliai sutapo su faktine reikšme.

Paklaidų histograma padeda patikrinti, ar klaidos daugiausia telkiasi netoli nulio ir ar matoma ryški asimetrija į vieną pusę.

In [ ]:
signed_error = y_true - y_pred

plt.figure(figsize=(9, 4))
plt.hist(signed_error, bins=30)
plt.axvline(0, linestyle="--", label="Nulinė paklaida")
plt.xlabel(r"Paklaida $e_i = y_i - \hat{y}_i$")
plt.ylabel("Stebėjimų skaičius")
plt.title("MLP prognozavimo paklaidų pasiskirstymas")
plt.legend()
plt.tight_layout()
plt.show()

mean_error = float(np.mean(signed_error))
median_error = float(np.median(signed_error))

if mean_error > 0:
    direction = "vidutinė paklaida yra teigiama, todėl šiame DEV skaidinyje matomas polinkis paklausą neįvertinti"
elif mean_error < 0:
    direction = "vidutinė paklaida yra neigiama, todėl šiame DEV skaidinyje matomas polinkis paklausą pervertinti"
else:
    direction = "vidutinė paklaida lygi nuliui"

display(Markdown(f'''
### Paklaidų pasiskirstymo santrauka

- vidutinė pasirašyta paklaida: **{mean_error:.2f}**;
- medianinė pasirašyta paklaida: **{median_error:.2f}**;
- pagal vidutinės paklaidos ženklą: **{direction}**.

Ši išvada taikoma analizuojamam DEV skaidiniui ir nėra automatiškai perkeliama galutiniam TEST laikotarpiui.
'''))

## 13. Ką šie papildomi bandymai pagrindžia?

Šie grafikai nėra naudojami naujiems hiperparametrams parinkti. Jie skirti jau pasirinktos MLP konfigūracijos elgsenai paaiškinti.

Kartu jie suteikia kelis skirtingus įrodymus:

- **mokymosi kreivė** parodo modelio optimizavimo eigą;
- **realiai atliktų iteracijų skaičius** parodo, kaip veikė nustatyta mokymo procedūra;
- **faktinės ir prognozuotos paklausos sklaidos diagrama** parodo prognozių atitikimą faktinėms reikšmėms;
- **paklaidų pasiskirstymas** padeda nustatyti prognozavimo klaidų kryptį;
- **MAE, RMSE, Peak MAE ir PUR** kiekybiškai įvertina prognozavimo tikslumą;
- **palyginimas su sezoniniu atskaitos metodu** parodo, ar sudėtingesnio MLP taikymas DEV rinkinyje suteikė išmatuojamą naudą.

Todėl MLP pasirinkimas ir vertinimas grindžiamas ne vien galutine metrikos reikšme, bet visa nuoseklia eksperimento eiga.

## 14. Klaidų analizė

Absoliuti prognozavimo klaida:

$$
e_i^{\mathrm{abs}}=\left|y_i-\hat{y}_i\right|
$$

čia:

- $y_i$ – faktinė paklausa;
- $\hat{y}_i$ – MLP prognozė;
- $e_i^{\mathrm{abs}}$ – absoliuti klaida.

Toliau pateikiama 10 didžiausių paskutinio DEV skaidinio klaidų ir vidutinė absoliuti klaida pagal paros valandą.

In [ ]:
error_df=pd.DataFrame({
    "Laikas":va["target_time"].to_numpy(),
    "Valanda":va["hr"].to_numpy(),
    "Faktinė paklausa":va["target"].to_numpy(),
    "MLP prognozė":pred
})
error_df["Absoliuti klaida"]=np.abs(error_df["Faktinė paklausa"]-error_df["MLP prognozė"])
error_df["Kryptis"]=np.where(
    error_df["MLP prognozė"]<error_df["Faktinė paklausa"],
    "Prognozuota per mažai","Prognozuota per daug"
)
display(error_df.nlargest(10,"Absoliuti klaida").reset_index(drop=True))

hour_error=error_df.groupby("Valanda")["Absoliuti klaida"].mean()
plt.figure(figsize=(10,4))
plt.bar(hour_error.index,hour_error.values)
plt.title("MLP vidutinė absoliuti klaida pagal paros valandą")
plt.xlabel("Valanda"); plt.ylabel("Vidutinė absoliuti klaida")
plt.xticks(range(24)); plt.tight_layout(); plt.show()

worst_hour=int(hour_error.idxmax())
display(Markdown(f'''
### Klaidų analizės santrauka

Didžiausia vidutinė absoliuti klaida šiame DEV skaidinyje gauta **{worst_hour}:00 val.** – **{hour_error.loc[worst_hour]:.2f}** nuomos.

Didžiausių klaidų lentelėje taip pat matoma klaidos kryptis – ar paklausa prognozuota per maža, ar per didelė.
'''))

## 15. Praktinė prognozavimo klaidų reikšmė

Šio uždavinio tikslas yra ne vien sumažinti statistinę klaidą. Prognozė turi padėti dviračių dalijimosi sistemos operatoriui planuoti dviračių perskirstymą.

Jeigu paklausa prognozuojama **per maža**, tam tikroje vietoje ar laikotarpyje gali būti suplanuotas nepakankamas dviračių kiekis. Piko metu tai ypač svarbu, todėl atskirai vertinamos Peak MAE ir PUR metrikos.

Jeigu paklausa prognozuojama **per didelė**, operatorius gali numatyti didesnį perskirstymo poreikį, negu jo iš tikrųjų reikėjo.

Šiame darbe šios pasekmės kiekybiškai nevertinamos pinigine išraiška. Jos naudojamos rezultatų praktinei reikšmei paaiškinti.

## 16. MLP eksperimento išvada

Šiame užrašinėje:

1. MLP formulės susietos su konkrečiais programinio kodo parametrais;
2. atlikta platesnė 12 konfigūracijų hiperparametrų paieška;
3. pagal realų geriausią DEV rezultatą atliktas antras, siauresnis hiperparametrų tikslinimo etapas;
4. visos konfigūracijos vertintos tais pačiais chronologiniais DEV skaidiniais ir metrikomis;
5. galutinė MLP konfigūracija pasirinkta tik pagal DEV;
6. TEST hiperparametrų paieškai nenaudotas;
7. pateiktas konfigūracijos stabilumas, faktinės ir prognozuotos paklausos grafikas bei konkrečių klaidų analizė.

Jei antrasis paieškos etapas nepagerina MAE, tai taip pat yra pagrįstas eksperimento rezultatas – tikrintoje siauresnėje parametrų srityje geresnė konfigūracija nebuvo rasta.

### Ribotumai

MLP yra jautrus požymių masteliui, tinklo struktūrai, reguliarizavimui, mokymosi spartai ir pradinei svorių inicializacijai. Atkuriamumui naudojama fiksuota `SEED` reikšmė. Vieno duomenų rinkinio rezultatai savaime neįrodo, kad ta pati konfigūracija būtų geriausia kitame mieste ar kitame laikotarpyje.

Abliacijos ir atsparumo bandymai šiame MLP užrašinėje nekartojami – jie atliekami atskirame pagrindinio tiriamo modelio eksperimente prieš galutinį TEST palyginimą.